In [ ]:
import importlib
for pkg in ["numpy", "matplotlib", "pyvista", "ipywidgets"]:
    importlib.import_module(pkg)
print("All libraries are installed.")

In [ ]:
from extra import *
from visualizations import *

# Lab 2: Single Spin Dynamics (MRI Physics)

Howdy! Welcome to the second programming lab of the MRI decal. In this lab, we're going to build a rudimentary bloch equation simulator. This is the differential equation that describes the movement of spins' magnetization, which as you can imagine, is pretty important in MR!

This one might be one of the longer labs, but I think it will be pretty cool. I've used several bloch simulators written by others, but I've always wanted to make a bloch simulator myself. So, as I begin to write this, I am feeling pretty excited. Hope you find it interesting!

## Part 1: Representing Vectors and Time

This is some preliminary material, which may be intuitive or already known by some of y'all, but which is nonetheless important to cover before we even _try_ to simulate the magnetization vector. Alrighty, first up, ___vectors___.

What is a vector? There are a few definitions. I think a physicist would answer "something that transforms like a vector" but I don't even know what the heck this means$^1$. Vector Perkins would say it's got magnitude and direction, and he'd be right also. But for our simulation it will be useful to consider a vector (say, $\mathbf{M}$) in terms of its _components_ (say, $M_x$, $M_y$, $M_z$). Each component is going to tell you how "big" the vector is, in that direction.

Your first task is going to be to define *three* vectors: one that points along the $x$ axis, one that lives in the $y-z$ plane, and then a third which is perpendicular to the first two. _Hint_: you can do this by thinking, but it may be more useful to remember the vector [cross product](https://en.wikipedia.org/wiki/Cross_product) which is implemented by numpy with `np.cross`.

Then we'll plot them to make sure the result looks right. Feel free to play around with the vectors you choose to plot, just to see what they look like. The window is interactive, so you're able to look around and zoom in / out. The red one should point with $\hat x$, the cyan one should be in the $y-z$ plane, and the violet one should look orthogonal to the other two.

$^1 \color{violet}\text{vaughan u should write a footnote here lol.}$

In [ ]:
v_x_ax = # TODO: vector that points in the x direction
v_yz_p = # TODO: write a vector in the y-z plane
v_perp = # TODO: find a vector perpendicular to the previous two

vecs = np.array([v_x_ax, v_yz_p, v_perp])
colors = np.array(['red', 'cyan', 'violet'])
plot_3D_vectors(vecs, colors=colors)

If your notebook starts to slow down big-time, you can go back and clear the output of these 3D visualization cells. I don't know how computationally taxing it is to keep them open. My laptop could handle it just fine, but your mileage may vary.

Remember, from the last lab, that we can represent rotations with complex numbers? Well, you can also represent rotations with _matrices_, which (in one view) are blocks of numbers which transform vectors. Check out the [wiki page](https://en.wikipedia.org/wiki/Rotation_matrix) about rotation matrices if you want to learn more.

Fill in the following function that returns a matrix which rotates a vector around the $y$ axis by an angle $\theta$.

By the way, matrix multiplication with numpy looks like this: `A @ B`.

In [ ]:
# TODO: Implement this function.
# Returns a (3, 3) numpy array
def y_rotation_matrix(theta):
    row1 =
    row2 =
    row3 =
    return np.vstack((row1, row2, row3))

In [ ]:
# gut check: choose a vector and theta by which to rotate it. if you peer down
# the y axis, the vector should look like it's rotated counter-clockwise by theta
theta = # TODO
v = # TODO
v_rot = y_rotation_matrix(theta) @ v

vecs = np.array([v, v_rot])
colors = np.array(['red', 'blue'])
labels = np.array(['original', 'rotated'])
plot_3D_vectors(vecs, colors=colors, labels=labels)

Next up, we're going to define some notion of ___time___. The whole point of these numerical simulations is that you can approximate the real, continuous behavior of something in _discrete time steps_, right? So, let's make some discrete time steps.

We can do this with `np.linspace`. Check out the [relevant documentation](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html) if you've never heard of it.

The problem is that it can be challenging to choose the _right_ interval `dt` between time steps. You want to balance computational complexity (small `dt`, more computation) with accuracy. Below, I want you to practice using `np.linspace` and then plot the `bruce` function, which returns $(x, y)$ coordinates parameterized by $t$ for different `dt`. You should be able to recognize the pictured result!

In [ ]:
duration = 1
dt = # TODO: determine a time interval dt that makes the plot look smooth.
num_steps = # TODO: calculate num_steps. this should be an integer.

# TODO: fill in the arguments to this function
t = np.linspace(_, _, _)
x, y = bruce(t)

plt.plot(x, y)
plt.axis('off')
plt.show()

The bloch equation (hold on - we will get to it eventually!) tells us that the derivative of a vector $\frac{d\mathbf M}{dt}$ is related in some way to $\mathbf M$. Let's now consider what it means to take the derivative of a _vector_.

Or maybe we should first think about what a derivative even _is_. Recall Taylor's Theorem:
$$
\mathbf M (t_0 + \Delta t) = \mathbf M (t_0) + \frac{d\mathbf M}{dt}(t_0) \Delta t + \frac{1}{2} \frac{d^2\mathbf M}{dt^2}(t_0) \Delta t ^2 + \cdots
$$
This says that the amount by which $\mathbf M$ changes with a small perturbation (that is, we added $\Delta t$) to its input is _approximately proportional_ to that perturbation. The constant of proportionality is the derivative.

Now, consider just one component. Intuitively, the amount by which _that_ coordinate changes with a small perturbation is exactly the derivative of _that_ coordinate with respect to the input. This means we can rewrite the above equation for each _component_, like this:
$$
M_z (t_0 + \Delta t) = M_z (t_0) + \frac{d M_z}{dt}(t_0) \Delta t + \frac{1}{2} \frac{d^2 M_z}{dt^2}(t_0) \Delta t ^2 + \cdots
$$
Your next task is to take the derivative of a vector. This should be simple enough - it's just a normal derivative on each component - but to do so, you'll also have to define the notion of a "discrete" derivative by ignoring the higher-order terms in the taylor expansion and solving for $\frac{d M_z}{dt}$.

Some notes on the conventions I chose:
+ If $\mathbf M$ is a three dimensional vector, we're going to store components down the *rows* and time across the *columns*. For example, this means `M[0, :]` will give you the first component for all time points and `M[:, 0]` will give you all three components for the first time point.
+ Remember that our sample points are spaced by $\Delta t$, so if $\mathbf M (t_0)$ is stored in `M[:, n]`, then $\mathbf M (t_0 + \Delta t)$ is stored in `M[:, n+1]`

In [ ]:
# TODO: Implement this function.
# Computes the discrete derivative, computed across the columns of
# M, since each row is a different vector component in our convention
def vector_derivative(M, dt):
    # HINT: because the discrete version of a derivative relies on taking
    #       the difference between consecutive time points, the returned
    #       vector dM/dt should be shorter than the original vector M
    # HINT: M[:, 1:] gives all the columns of M except the first, and
    #       M[:, :-1] gives all the columns of M except the last.
    return None

Let's do a quick gut-check. Recall from calculus some simple functions and their derivatives, and define $M_x$ and $M_y$ as these functions. Then, plot the components of $\mathbf M$ and $\frac{d \mathbf M}{dt}$ and make sure these plots match your expectation.

If you're feeling inclined to, it might be interesting to go back a few cells, change `dt`, and see if the plots still look ok.

In [ ]:
# TODO: define the components Mx and My to be functions you know the derivative of
M_x, M_y = _, _
# we'll stack Mx and My into a vector where:
#   each row is a component
#   each column gives a point in time
M = np.vstack((M_x, M_y))

plot_vector_components(M, t, title=r'$\mathbf{M}$ components over time')
plot_vector_components(vector_derivative(M, dt), t[:-1],
                       title=r'$\frac{d \mathbf{M}}{dt}$ components over time')

## Part 2: Simulating $T_1$ Recovery
Ok, I think that now is a good time to introduce _part_ of the bloch equation. Consider just the $M_z$ component, which experiences something called "$T_1$ recovery" governed by the following differential equation:
$$
\frac{dM_z}{dt} = \frac{M_0 - M_z}{T_1}
$$
Using this equation, we can use the Taylor expansion to approximate $M_z(t + \Delta t)$. The approach is very similar to what you did when we were talking about the whole "discrete derivative" thing. Write the following function to do this.

In [ ]:
# TODO: Implement this function.
# Given Mz at time t, computes and returns Mz at time
# t + dt according to the T1 recovery differential equation
def T1_update_Mz(Mz_current, M0, T1, dt):
    return None

Now, how can we use this to figure out $M_z$ for _any_ time $t$? We define $M_z(0)$, which is all the simulator needs to determine $M(\Delta t)$, which is all the simulator needs to determine $M(2 \Delta t)$, and so on. This reminds _me_ of lining up and knocking down dominos; we push over the first domino by telling the simulator how to start, and then each one pushes over the next until all the dominos are knocked over and we know $M(t)$.

In [ ]:
# TODO: Implement this function.
# Given an initial value of Mz, computes Mz at the next
# num_steps consecutive time points, each spaced by dt.
def simulate_T1_recovery(Mz_initial, M0, T1, dt, num_steps):
    # HINT: if you know Mz[i], T1_update_Mz gives you Mz[i+1]
    Mz = np.zeros(num_steps)
    Mz[0] = ____
    for step in range(num_steps - 1):
        Mz[step + 1] = ____
    return None

This nice thing is that this particular differential equation has an exact solution:
$$
M_z(t) = M_0 (1 - e^{-t / T_1}) + M_z(0) e^{-t / T_1}
$$
You can check that this is true by plugging it back into the differential equation if you want to.

We're next going to plot our simulated recovery next to the exact curve. If `dt` is small enough, the curves should be very close!

In [ ]:
Mz_initial = 2
M0 = 5
T1 = 1

dt = 0.01
duration = 12
num_steps = int(duration/dt) + 1
t = np.linspace(0, duration, num_steps)

Mz_exact = # TODO: write the exact solution
Mz_simulated = simulate_T1_recovery(Mz_initial, M0, T1, dt, num_steps)

# treat the two simulations as vector components for the sake of using the same plot method
plot_vector_components(np.array((Mz_exact, Mz_simulated)), t,
                       labels=['Exact', 'Simulated'], linestyles=('b-', 'y--'))

It would be interesting to, based on the exact solution, think about how much you would _expect_ $M_z$ to have recovered after time $T_1$, and then set `duration = T1` to see if you're right. After how long would you expect $M_z$ to approximately equal $M_0$, say $M_z(t) = 0.95M_0$? Based on this analysis (and by testing different values in the cell above), answer the following question:

If a spin has a longer $T_1$, does it take _longer_ or _shorter_ for that spin to recover its longitudinal relaxation?

<span style="color:cyan">
Your answer here.
</span>

## Part 3: Simulating $T_2$ Decay

Take a look at the differential equation for the transverse component $M_{xy}$:
$$
\frac{dM_{xy}}{dt} = -\frac{M_{xy}}{T_2}
$$
I hope you're thinking that this part should be _quite similar_ to the last. I mean, if we assume that $M_{xy}$ is just a number, just like $M_{z}$, the first step is _identical_ to the previous part...

In [ ]:
# TODO: Implement this function.
# Given Mxy at time t, computes and returns Mxy at time
# t + dt according to the T2 decay differential equation
def T2_update_Mxy(Mxy_current, T2, dt):
    return None

Convince yourself that $M_{xy}$ is unlike the longitudinal component in that it has _direction_. The transverse component could be pointing in $x$ or $y$ or somewhere in between, and it's still transverse. Hence you _could_ say that $M_{xy}$ is _itself_ a vector, but we don't, because the MRI signal is proportional only to its magnitude.

Anyway, when we update $M_{xy}$, we'll want to change its _magnitude only_. Its direction will start changing when we consider precession in the next part.

For now, let's consider only $|M_{xy}|$ and simulate the same solution curve as before. Fortunately, this also has a "ground truth" solution:
$$
M_{xy}(t) = M_{xy}(0) e^{-t / T_2}
$$
I would recommend copy-and-pasting the two analogous code cells you wrote in part 2, and changing what needs to be changed to make it work for this part. _Being able to copy-paste-tweak is a sometimes valuable skill :)_

In [ ]:
# TODO: Implement this function.
# Given an initial value of |Mxy|, computes |Mxy| at the next
# num_steps consecutive time points, each spaced by dt.
def simulate_T2_decay(Mxy_initial, T2, dt, num_steps):
    # HINT: if you know Mxy[i], T2_update_Mxy gives you Mxy[i+1]
    return None

In [ ]:
# TODO: come up with some simulation parameters
Mxy_initial = ____
T2 = ____

dt = 0.01
duration = 12
num_steps = int(duration/dt) + 1
t = np.linspace(0, duration, num_steps)

Mxy_exact = # TODO: write the exact solution
Mxy_simulated = # TODO: write the simulated solution

# treat the two simulations as vector components for the sake of using the same plot method
plot_vector_components(np.array((Mxy_exact, Mxy_simulated)), t,
                       labels=['Exact', 'Simulated'], linestyles=('b-', 'y--'))

Hopefully it's clear now why I like to call one of these relaxation processes 'recovery' and the other 'decay.' Ultimately, they're both called _relaxation_ because these are the mechanisms by which our perturbed magnetization returns to its polarized, low-energy, _relaxed_ state.

## Part 4: $T_1$ and $T_2$

Alright, now I want to put together $T_1$ and $T_2$ and see what they do to a _three dimensional_ magnetization vector. If we have the initial vector $\mathbf{M} (0)$ and we know the relaxed magnetization magnitude $M_0$, our update step should look something like this:

1) determine the scalars $M_z(t)$ and $M_{xy}(t)$
2) determine the new scalars $M_z(t + \Delta t)$ and $M_{xy}(t + \Delta t)$
3) create the new magnetization vector for time $t + \Delta t$ and add it to the array representation of $\mathbf{M}$.

Remember that you should be updating the _magnitude only_ of $M_{xy}$, and that we are representing $\mathbf{M}$ as a $3 \times \text{N}$ array, where the rows ($3$) are the three vector components and the columns ($\text{N}$) are the magnetization at $\text{N}$ different times, separated by $\Delta t$.

In [ ]:
# TODO: Implement this function.
# Compute the magnetization vector at num_steps points in
# time separated by dt, using the forward euler method.
def simulate_T1_and_T2(M_initial, M0, T1, T2, dt, num_steps):

    M = np.zeros((3, num_steps))
    M[:, 0] = # hint: this is the magnetization vector at time 0

    for step in range(num_steps - 1):

        # do the Mz update first cause it's easier
        Mz_current = ____
        Mz_next = T1_update_Mz(__, __, __, __)
        ____ = Mz_next

        # do the Mxy update next cause it gots to get done
        # HINT: write Mxy as a complex number for ease of life. this way,
        # you can easily get its magnitude with np.abs
        c_Mxy_current = ____ + 1j*____
        Mxy_current = np.abs(c_Mxy_current)
        Mxy_next = ____

        # HINT: you can get the old phase (direction) with np.angle and make a new
        # complex Mxy vector with the same phase but different magnitude.
        phase = ____
        c_Mxy_next = ____ * np.exp(1j*phase)

        # your final step should be to convert from the
        # complex form back to cartesian coordinates
        M[__, step+1] = np.real(c_Mxy_next)
        M[__, step+1] = np.imag(c_Mxy_next)

    return M

Let's plot all three components over time.

You should see a $T_1$-recovery-looking curve in the z component; and $T_2$-decay-looking curves in the $x$ and $y$ components.

In [ ]:
M_initial = [2*np.sqrt(2)/2, 2*np.sqrt(2)/2, 0]
M0 = 2
T1 = 2
T2 = 1

dt = 0.01
duration = 5
num_steps = int(duration/dt) + 1
t = np.linspace(0, duration, num_steps)

M = simulate_T1_and_T2(M_initial, M0, T1, T2, dt, num_steps)
plot_vector_components(M, t, linestyles=('b-', 'r-.', 'k-'))

The function below should animate $\mathbf M$'s motion. I limited the fps to 15 because my laptop could handle only 15 fps, but feel free to fiddle with that. Or, you can just move the frame slider around to see what the magnetization looks like at different static frames.

In [ ]:
animate_vector_motion(M, fps=15, dt=dt)

## Part 5: Simulating Precession

The final part of the simulation is going to be the consideration of _precession_. Recall that this is kinda-sorta the "meat and potatoes" of MRI, because the signal ends up being proportional to how fast the spins are spinning. You don't have to worry about that, though. Let's just make 'em ___spin___, yo!

If I were you, I'd take a break now, if you haven't yet. I told you it would be a long lab :)

The precession-inducing part of the bloch equation goes like this:
$$
\frac{d \mathbf M}{dt} = \gamma (\mathbf M \times \mathbf B)
$$
For simplicity, let's assume that $\mathbf B$ is pointing with $\hat z$. Why? Because dealing with an arbitrary rotation axis is hard and I don't feel like writing that for this lab. Maybe next time...

Anyway, this is actually pretty accurate for an MRI scanner too (mostly... other fields turn on only during the excitation phase). First, let's make a rotation matrix that spins a vector about $\hat z$. This is your chance to go back and review part 1, where you made a rotation matrix about $\hat y$! Feel free to copy-and-paste into a new cell, some of the visualization code we used back there, if you want to visually check your work.

In [ ]:
# TODO: Implement this function.
# Returns a (3, 3) numpy array
def z_rotation_matrix(theta):
    return None

Next let's write a function that will _apply_ the rotation to $\mathbf M$.

In [ ]:
# TODO: Implement this function.
# Given M at time t, computes and returns M at time
# t + dt according to the bloch rotation solution
# assuming a rotation around the z axis
def rotate_update_M(M, theta):
    return None

Next we're going to write a function that can calculate the amount of rotation that should be applied at each time step. Here are some facts you can use to "derive" this and write your function:

+ the precession frequency is $|\omega_0| = \gamma |B|$
+ the vector precesses clockwise, so your $\theta$ updates should be negative
+ so in time $\Delta t$, the vector rotates by $-|\omega_0| \Delta t$

In [ ]:
# TODO: Implement this function.
# Returns by how much to rotate a vector in time dt
def get_theta(B, dt, gamma=2*np.pi*4.2577e3):
    return None

Note that $\gamma$ is really big: $4.2577 \text{kHz}/\text{G}$, to be exact. Suppose $B = 15000 \text{G}$, which is the magnetic field in the $1.5 \text{T}$ aspect scanner we'll use later in this class. Our vector is going to precess... how many times per second?

<span style="color:cyan">
Your answer here.
</span>

Ok - now we can do something fun! Let's use these functions to simulate _precession only_ and visualize the results. Based on the previous answer, we're going to use a _much much lower field_ so that the magnetization doesn't spin too fast.

In [ ]:
# TODO: Implement this function.
# Given an initial value of M, computes M at the next
# num_steps consecutive time points, each spaced by dt.
# Considers only the effect of precession.
def simulate_precession(M_initial, B, dt, num_steps):
    # HINT: M should be a (3xnum_steps) array
    # HINT: if you know M[i], rotate_update_M gives you M[i+1]
    M = ____
    M[:, 0] = ____
    for step in range(____):
        theta = ____
        M[:, step + 1] = ____
    return M

Perhaps you know what's coming...

Precession _also_ has an analytical solution against which we can check our simulation! I'll write it component by component. It is:
$$
M_x(t) = M_x(0) \cos \omega_0 t - M_y(0) \sin \omega_0 t
$$
$$
M_y(t) = M_x(0) \sin \omega_0 t + M_y(0) \cos \omega_0 t
$$
$$
M_z(t) = M_z(0)
$$
where, remember, $\omega_0 = -\gamma B$. Beware that we are plotting _a lot_ of components so the plot may look a wee bit messy.

In [ ]:
M_initial = np.array([1/np.sqrt(3), 1/np.sqrt(3), 1/np.sqrt(3)])
B = 1e-4 # gauss
gamma = 2*np.pi*4.2577e3 # radians / gauss

dt = 0.01
duration = 5
num_steps = int(duration/dt) + 1
t = np.linspace(0, duration, num_steps)

omega_0 = # TODO: write the precession frequency
Mx = M_initial[0] * np.cos(omega_0 * t) - M_initial[1] * np.sin(omega_0 * t)
My = M_initial[0] * np.sin(omega_0 * t) + M_initial[1] * np.cos(omega_0 * t)
Mz = np.full(len(t), M_initial[2, None])
M_exact = np.vstack((Mx, My, Mz))
M_simulated = # TODO: use your simulation function

# stack all six components for the visual check
components = np.vstack((M_exact, M_simulated))
labels = ['$x$, Exact', '$y$, Exact', '$z$, Exact',
          '$x$, Simulated', '$y$, Simulated', '$z$, Simulated']
linestyles = ['b-', 'r-', 'g-', 'c-.', 'm-.', 'y-.']
plot_vector_components(components, t, labels=labels, linestyles=linestyles)

You're getting to know me pretty well through these labs, maybe. Let's see if we can finish each other's sentences:

As expected, the transverse $\hat x$ and $\hat y$ components look like
<span style="color:cyan">
{your first answer here}
</span>
and the longitudinal $\hat z$ component
<span style="color:cyan">
{your second answer here}
</span>. <br>

Alright, alright, I hear ya. We'll animate it... you should be able to _see_ the vector spinning in a clockwise direction around the $\hat z$ axis.

In [ ]:
animate_vector_motion(M_simulated, fps=24, dts_per_frame=3, dt=dt, show_path=False)

## Part 6: Putting it All Together

In this final part, the grand finale, we do it ___all___: decay, recovery, _and_ precession! Let's get 'er done.

Finish the function below. I'd copy-and-paste the $T_1$ and $T_2$ simulation code you wrote earlier and add some precession after applying the decay and recovery. Just remember to rotate the _same vector_ that got relaxed; otherwise, you'll be overwriting all the relaxation and the result won't look right.

In [ ]:
# TODO: Implement this function.
# Compute the magnetization vector at num_steps points in
# time separated by dt, using the forward euler method.
def simulate_all(M_initial, M0, B, T1, T2, dt, num_steps):
    return None

I don't want to bother with plotting the analytic, ground truth solution too. I'm sure you agree. This lab is getting _prett dang_ long. If you did the other parts you should be feeling pretty confident that everything works anyway. Let's simulate!

In [ ]:
M_initial = [2*np.sqrt(2)/2, 2*np.sqrt(2)/2, 0]
M0 = 2
B = 1e-3
T1 = 2
T2 = 1

dt = 0.01
duration = 5
num_steps = int(duration/dt) + 1
t = np.linspace(0, duration, num_steps)

M = # TODO: use your simulation function
plot_vector_components(M, t, linestyles=('b-', 'r-.', 'k-'))

In [ ]:
animate_vector_motion(M, fps=10, dts_per_frame=2, dt=dt, show_path=True)

Ahhhh, that is __pretty cool__. Mess around with the parameters and see what happens. It's probably good to try to make a guess, as to what the tweak you make _should_ do, and then confirm or deny. That'll help ya learn.

Thanks for working through this assignment, and I hope you had as much fun doing it as I had writing it.

___this lab and accompanying visualization written by Ryan Black for the MRI decal, fall 2026___